In [2]:
%load_ext autoreload
%autoreload 2

from src.agent import run_agent

In [3]:
# Test 1 — Baseline (should use only search_book)

print("="*70 + "\nTEST 1: Baseline — single tool\n" + "="*70)
run_agent("What is retrieval augmented generation?")

TEST 1: Baseline — single tool

--- Turn 1 ---
Tool call: search_book({'query': 'retrieval augmented generation RAG'})
Tool returned 4730 chars

--- Turn 2 ---
Agent (final): Based on the AI Engineering book, **Retrieval Augmented Generation (RAG)** is a technique for improving AI model responses by dynamically retrieving relevant information from external data sources and providing it to the model.

## Key Concepts:

**The Core Problem It Solves:**
RAG addresses the challenge of knowledge-intensive tasks where all available knowledge can't be directly input into the model. Just as humans are more likely to give wrong answers when lacking information, AI models are prone to mistakes and hallucinations when they're missing context.

**How It Works:**
The RAG pattern follows a "retrieve-then-generate" approach:
1. A user makes a query
2. A retriever component finds the most relevant information from external data sources
3. This retrieved information is fed to the model as context
4. The

'Based on the AI Engineering book, **Retrieval Augmented Generation (RAG)** is a technique for improving AI model responses by dynamically retrieving relevant information from external data sources and providing it to the model.\n\n## Key Concepts:\n\n**The Core Problem It Solves:**\nRAG addresses the challenge of knowledge-intensive tasks where all available knowledge can\'t be directly input into the model. Just as humans are more likely to give wrong answers when lacking information, AI models are prone to mistakes and hallucinations when they\'re missing context.\n\n**How It Works:**\nThe RAG pattern follows a "retrieve-then-generate" approach:\n1. A user makes a query\n2. A retriever component finds the most relevant information from external data sources\n3. This retrieved information is fed to the model as context\n4. The model generates a response using both its instructions and the retrieved context\n\n**Key Benefits (from page 278):**\nAccording to the original RAG paper (Lew

In [7]:
# Test 2 — The Big One: Sequential Chaining

print("="*70 + "\nTEST 2: Search + summarize\n" + "="*70)
run_agent("Summarize what the book says about fine-tuning in 3 bullet points.")

TEST 2: Search + summarize

--- Turn 1 ---
Tool call: search_book({'query': 'fine-tuning', 'top_k': 5})
Tool returned 3632 chars

--- Turn 2 ---
Agent (final): Based on the search results from the AI Engineering book, here are 3 key bullet points about fine-tuning:

• **Evaluation Before Fine-Tuning**: The book emphasizes the importance of evaluation and quality assurance *before* starting fine-tuning, as it can be a costly process. This suggests a strategic approach where fine-tuning should not be the first solution considered (page 4).

• **Parameter-Efficient Fine-Tuning**: Early fine-tuning approaches required training entire models (full fine-tuning), which is memory-intensive. The book discusses parameter-efficient fine-tuning techniques that reduce memory requirements, making fine-tuning more accessible to practitioners with limited computational resources (pages 356-357).

• **Fine-Tuning vs. RAG for Specific Tasks**: The book references research showing that for tasks requirin

'Based on the search results from the AI Engineering book, here are 3 key bullet points about fine-tuning:\n\n• **Evaluation Before Fine-Tuning**: The book emphasizes the importance of evaluation and quality assurance *before* starting fine-tuning, as it can be a costly process. This suggests a strategic approach where fine-tuning should not be the first solution considered (page 4).\n\n• **Parameter-Efficient Fine-Tuning**: Early fine-tuning approaches required training entire models (full fine-tuning), which is memory-intensive. The book discusses parameter-efficient fine-tuning techniques that reduce memory requirements, making fine-tuning more accessible to practitioners with limited computational resources (pages 356-357).\n\n• **Fine-Tuning vs. RAG for Specific Tasks**: The book references research showing that for tasks requiring up-to-date information (like current events), Retrieval-Augmented Generation (RAG) can outperform fine-tuned models, suggesting fine-tuning may not alw

In [ ]:
# Test 2 — version 2, sharpen the tool description to force usage

print("="*70 + "\nTEST 2: Search + summarize\n" + "="*70)
run_agent("Summarize what the book says about fine-tuning in 3 bullet points.")

## Test 2 Observation
Model called search_book but skipped summarize_passage. It produced a summary directly from the retrieved context.

Interpretation: When the model is already capable of doing the 
downstream operation (summarizing text), a separate tool for that 
operation is redundant. The tool description ("Use AFTER search_book 
when user asks for summary") competes with the model's default 
behavior of just summarizing.

When would a separate summarizer tool actually help?
- Cost isolation: cheap model for summarization, expensive model for reasoning
- Prompt specialization: tight rubrics that would clutter main system prompt
- Reusability across multiple agents

For this project, summarize_passage is educational but not necessary.

In [5]:
# Test 3 — Extract Terms Path

print("="*70 + "\nTEST 3: Search + extract terms\n" + "="*70)
run_agent("What are the key technical terms in the RAG chapter?")

TEST 3: Search + extract terms

--- Turn 1 ---
Tool call: search_book({'query': 'RAG retrieval augmented generation technical terms', 'top_k': 5})
Tool returned 4507 chars

--- Turn 2 ---
Agent (final): Based on the search results, here are the **key technical terms in the RAG chapter**:

## Core RAG Concepts
- **RAG (Retrieval-Augmented Generation)** - The main technique of retrieving relevant information from external sources and augmenting the model's generation with it
- **Retrieve-then-generate pattern** - The workflow where retrieval happens first, then generation follows
- **Document reader** - The model component that processes retrieved documents

## Retrieval Algorithms
- **Term-based retrieval** - Traditional keyword-based search
- **Embedding-based retrieval** - Semantic search using vector embeddings
- **Sparse retrievers** - Term-based retrieval methods
- **Dense retrievers** - Embedding-based retrieval methods

## Retrieval Optimization Techniques
- **Chunking strategy**

"Based on the search results, here are the **key technical terms in the RAG chapter**:\n\n## Core RAG Concepts\n- **RAG (Retrieval-Augmented Generation)** - The main technique of retrieving relevant information from external sources and augmenting the model's generation with it\n- **Retrieve-then-generate pattern** - The workflow where retrieval happens first, then generation follows\n- **Document reader** - The model component that processes retrieved documents\n\n## Retrieval Algorithms\n- **Term-based retrieval** - Traditional keyword-based search\n- **Embedding-based retrieval** - Semantic search using vector embeddings\n- **Sparse retrievers** - Term-based retrieval methods\n- **Dense retrievers** - Embedding-based retrieval methods\n\n## Retrieval Optimization Techniques\n- **Chunking strategy** - How documents are divided into retrievable units\n- **Contextual retrieval** - Retrieving information with appropriate context\n- **Query rewriting** - Reformulating queries for better 

## Test 3 Observation
* only called "serach tool", not "extract term tool"
* extract key term tool transforms text model already has this function. So the model bypasses it. 
* The model isn't ignoring my tools out of confusion. It's making a reasonable efficiency call: don't bounce to a tool for something you can do inline.


In [3]:
# Test 4 — Tool Selection Under Ambiguity

print("="*70 + "\nTEST 4: Ambiguous — which tool to use?\n" + "="*70)
run_agent("Give me a brief overview of evaluation methods.")

TEST 4: Ambiguous — which tool to use?

--- Turn 1 ---
Tool call: search_book({'query': 'evaluation methods', 'top_k': 5})
Tool returned 5064 chars

--- Turn 2 ---
Agent (final): Based on the AI Engineering book, here's a brief overview of evaluation methods:

## Overview of Evaluation Methods

The book emphasizes that **evaluation is critical and complex**, with dedicated coverage due to its importance. Here are the key aspects:

### Purpose
Evaluation aims to **mitigate risks and uncover opportunities** by identifying where your system is likely to fail and designing evaluation around those failure points.

### Main Evaluation Methods
Different evaluation criteria may require different methods, including:
- **Specialized classifiers** (e.g., toxicity detection)
- **Similarity measures** (e.g., semantic similarity for relevance)
- **AI judges** (to measure aspects like factual consistency)
- **Language modeling metrics** (like text perplexity)
- **Exact evaluation** (measuring functio

"Based on the AI Engineering book, here's a brief overview of evaluation methods:\n\n## Overview of Evaluation Methods\n\nThe book emphasizes that **evaluation is critical and complex**, with dedicated coverage due to its importance. Here are the key aspects:\n\n### Purpose\nEvaluation aims to **mitigate risks and uncover opportunities** by identifying where your system is likely to fail and designing evaluation around those failure points.\n\n### Main Evaluation Methods\nDifferent evaluation criteria may require different methods, including:\n- **Specialized classifiers** (e.g., toxicity detection)\n- **Similarity measures** (e.g., semantic similarity for relevance)\n- **AI judges** (to measure aspects like factual consistency)\n- **Language modeling metrics** (like text perplexity)\n- **Exact evaluation** (measuring functional correctness and similarity against reference data)\n\n### Flexible Approach\nYou can **mix and match evaluation methods** for the same criteria. For example, y

In [4]:
# Test 5 — No Tools Needed

print("="*70 + "\nTEST 5: Casual — should skip tools\n" + "="*70)
run_agent("Hi, what can you help me with today?")

TEST 5: Casual — should skip tools

--- Turn 1 ---
Agent (final): Hi! I'm here to help you learn about **AI Engineering**. I have access to a comprehensive book on the subject and can assist you in several ways:

**I can help you:**
- **Find specific concepts or techniques** - Ask about particular AI engineering topics, methods, or terminology
- **Get an overview** - Understand what major topics are covered in the book
- **Search for information** - Look up passages related to your questions
- **Learn about best practices** - Discover practical approaches to AI engineering challenges

**Some example questions I can answer:**
- "What is prompt engineering?" or "How do I handle data validation in ML pipelines?"
- "What does this book cover?" for a high-level overview
- "Explain [specific AI engineering concept]"
- "How do I approach [specific AI engineering problem]?"

Feel free to ask me anything related to AI engineering, and I'll search the book to give you accurate, detailed answers!

'Hi! I\'m here to help you learn about **AI Engineering**. I have access to a comprehensive book on the subject and can assist you in several ways:\n\n**I can help you:**\n- **Find specific concepts or techniques** - Ask about particular AI engineering topics, methods, or terminology\n- **Get an overview** - Understand what major topics are covered in the book\n- **Search for information** - Look up passages related to your questions\n- **Learn about best practices** - Discover practical approaches to AI engineering challenges\n\n**Some example questions I can answer:**\n- "What is prompt engineering?" or "How do I handle data validation in ML pipelines?"\n- "What does this book cover?" for a high-level overview\n- "Explain [specific AI engineering concept]"\n- "How do I approach [specific AI engineering problem]?"\n\nFeel free to ask me anything related to AI engineering, and I\'ll search the book to give you accurate, detailed answers! What would you like to know?'

In [5]:
# Test 6 — Compound Query

print("="*70 + "\nTEST 6: Compound — chain of 3?\n" + "="*70)
run_agent(
    "Summarize the section on agents, and then give me the key terms it uses."
)

TEST 6: Compound — chain of 3?

--- Turn 1 ---
Tool call: search_book({'query': 'agents', 'top_k': 5})
Tool returned 3551 chars

--- Turn 2 ---
Agent (final): Based on the search results, here's a summary of the section on agents:

## Summary of Agents Section

The agents section (Chapter 6) discusses how foundation models have enabled the development of autonomous, intelligent agents that can act as assistants, coworkers, and coaches. According to the book, an **agent** is defined as anything that can perceive its environment through sensors and act upon it through actuators (page 300).

The key themes covered are:

1. **Emerging Field**: AI-powered agents are an emerging field with no established theoretical frameworks yet, so the book attempts to build a framework from existing literature (page 300).

2. **Applications**: Agents can help with diverse tasks like creating websites, gathering data, planning trips, conducting market research, managing accounts, automating data entry, co

"Based on the search results, here's a summary of the section on agents:\n\n## Summary of Agents Section\n\nThe agents section (Chapter 6) discusses how foundation models have enabled the development of autonomous, intelligent agents that can act as assistants, coworkers, and coaches. According to the book, an **agent** is defined as anything that can perceive its environment through sensors and act upon it through actuators (page 300).\n\nThe key themes covered are:\n\n1. **Emerging Field**: AI-powered agents are an emerging field with no established theoretical frameworks yet, so the book attempts to build a framework from existing literature (page 300).\n\n2. **Applications**: Agents can help with diverse tasks like creating websites, gathering data, planning trips, conducting market research, managing accounts, automating data entry, conducting interviews, and negotiating deals—with enormous potential economic value (page 299).\n\n3. **Capabilities**: Agent capabilities are determi

# Week 3 Day 1 — Sequential Tools Observations

## Test 2: Did the model chain search_book → summarize_passage?
No. The model only called search book and did the summarize inline without calling tool.

## Test 3: Did the model chain search_book → extract_key_terms?
No. The model only called search book and did the summarize inline without calling tool.

## Test 4: For the ambiguous "brief overview" query, which path did the model take?
The model called search_book. It didn't call summarize_passage. The model has the chunks. "Brief overview" is a synthesis task the model can do inline. No downstream tool needed. Answer directly.

## Test 6: For the compound query, how many tool calls did the model make?
The model only made one tool call serach tool. Model's actual behavior: search once, do both inline. The model made the decision because it's cheaper to retrieve broadly once than narrowly twice. 

Interview version: "I noticed my agent compressed a compound question into a single retrieval instead of decomposing it. That's the model doing implicit planning to minimize expensive operations."

Interview version: "Tool descriptions are prompts, not docstrings. I found I could change agent behavior significantly just by rewording the description without changing any code."

## Surprises
Tools chain when each tool does something the previous one can't do, and something the model can't do inline.
There is gap between tool design intent and the model's execution plan. So we apply LangGraph.

## meta-learning
The gap between my tool design intent and the model's execution plan is where real agent engineering happens.